# NanoHarness Technical Documentation

This notebook documents the current internal structure of NanoHarness for development work.

## Architecture Overview

NanoHarness is split into small modules:

- `main.py` runs startup checks, then launches the harness.
- `helpers/startup.py` creates the app directory, sessions directory, and starter config.
- `core/harness.py` owns app lifecycle, config, sessions, slash commands, and the main loop.
- `core/agent.py` owns model streaming, reasoning display, cancellation, permissions, and tool orchestration.
- `core/auditor.py` owns command risk classification and auditor-model explanations.
- `core/tui.py` owns prompt input and prompt history.
- `tools/common.py` owns shared path, truncation, and integer-clamping helpers.
- `tools/bash.py` owns the Bash tool schema and command execution.
- `tools/filesystem.py` owns structured file tools.
- `tools/http.py` owns the GET-only URL fetch tool.
- `tools/registry.py` exports the combined tool schema list and structured tool handlers.
- `core/paths.py` owns app paths.

## Startup Flow

`main.py` calls `startup_checks()` before starting the app. This guarantees app state exists before runtime code needs it.

```python
from helpers.startup import startup_checks
from core.harness import run

if __name__ == "__main__":
    startup_checks()
    run()
```

## App Paths

Runtime files are not stored in the repository by default. `core/paths.py` defines:

```python
APP_DIR = os.environ.get("NANOHARNESS_HOME", os.path.expanduser("~/.nanoharness"))
CONFIG_PATH = os.environ.get("NANOHARNESS_CONFIG", os.path.join(APP_DIR, "config.json"))
SESSIONS_DIR = os.path.join(APP_DIR, "sessions")
```

Startup ensures the primary config exists.

## Working Directory Resolution

Config includes `working_directory`, defaulting to `.`. This preserves launch-directory behavior while making the command base explicit.

Rules:

- If `working_directory` is missing, empty, or invalid, NanoHarness gracefully falls back to the launch directory.
- If `working_directory` is relative, it resolves from the launch directory.
- Bash commands run from the resolved configured working directory unless the model provides `cwd`.
- If model-provided `cwd` is relative, it resolves from the configured working directory.
- `/cwd` shows launch/configured/resolved paths.
- `/cwd set <path>` validates and persists a new configured working directory.

## Config Generation

`helpers/startup.py` contains a hardcoded `REFERENCE_CONFIG` and `CONFIG_VERSION`. If `CONFIG_PATH` does not exist, startup writes that reference to disk. If config exists, startup merges missing fields from the reference config and updates `config_version` without pruning unknown user keys.

Invalid JSON or non-object config files are backed up to `config.json.invalid.<timestamp>` and regenerated from the reference config. The config includes model settings, auditor model, Bash limits, and tool policy.

## Harness Responsibilities

`core/harness.py` is the app wrapper around the agent. It handles:

- Loading and saving config.
- Session creation, loading, deletion, and listing.
- Slash command handling.
- API key management via `/apikey`.
- OpenAI-compatible client creation.
- Main prompt loop.
- Error recovery after failed model turns.

## Agent Responsibilities

`core/agent.py` is limited to agent behavior:

- System prompt.
- Policy lookup and permission prompts.
- Streaming Chat Completions handling.
- Reasoning delta extraction and cleanup.
- esc/ctrl-c cancellation during streaming.
- Streaming tool-call accumulation.
- Bash tool auditing, user approval, and execution orchestration.
- Direct permission handling for structured filesystem and HTTP tools.

## Tool Registry

NanoHarness exposes several tools to the model through `tools/registry.py`:

- `bash` from `tools/bash.py`
- `list_files`, `read_file`, and `write_file` from `tools/filesystem.py`
- `fetch_url` from `tools/http.py`
- `web_search` from `tools/search.py`
- `read_document` from `tools/document.py`

`TOOLS` is the combined schema list passed to the model. `STRUCTURED_TOOL_HANDLERS` maps non-Bash tools to Python handlers. Bash is special-cased in `core/agent.py` because it uses the auditor approval flow.

Startup config policies now include `bash`, `list_files`, `read_file`, `write_file`, `fetch_url`, `read_document`, and `web_search`; missing policy keys are merged into existing configs by startup schema repair.

New code should import from the specific tool module or from `tools/registry.py`. The old `tools/tools.py` compatibility shim was removed after the split.

## Bash Tool

`tools/bash.py` defines the Bash schema and `run_bash()`. It supports either a single `command` string or a `commands` list. When `commands` is used, NanoHarness builds one Bash script and defaults to `stop_on_error: true`, prepending `set -e`.

The command is executed with:

```python
subprocess.run(
    command,
    shell=True,
    executable="/bin/bash",
    cwd=resolved_cwd,
    capture_output=True,
    text=True,
    timeout=timeout_seconds,
)
```

Output is returned as JSON with stdout/stderr truncation flags, exit code, resolved cwd, timeout, and output cap metadata.

## Filesystem Tools

`tools/filesystem.py` implements structured filesystem tools. These do not use the auditor model; they go through direct policy permission prompts in `core/agent.py`.

- `list_files` lists directory entries. It is non-recursive by default and caps returned entries.
- `read_file` reads bounded text from a file with `offset` and `max_chars`. It rejects directories and uses UTF-8 with replacement for decode errors.
- `write_file` supports `create`, `overwrite`, `append`, and `replace`. Replace mode uses `old_text` and `new_text`; it requires exactly one match unless `replace_all` is true.

All filesystem paths resolve from the configured working directory unless an absolute or `~/` path is supplied.

## Document Tool

`tools/document.py` implements `read_document`. It uses `pypdf` for text-based PDFs and ordinary text decoding for `.txt`, `.md`, `.csv`, `.json`, and `.log` files.

For PDFs, the tool accepts `page_start`, `page_count`, and `max_chars`. Page numbers are 1-based. `page_count` defaults to 5 and is capped at 25.

The tool does not OCR scanned or image-based PDFs. If no text is extracted, it returns a notice explaining that the PDF may be scanned. Document output includes an untrusted-content warning because documents can contain prompt injection.

## Web Search Tool

`tools/search.py` implements `web_search` using Python built-in libraries against DuckDuckGo's HTML endpoint. It is best-effort HTML parsing, not a guaranteed stable search API.

The tool accepts `query` and `max_results`, caps results at 10, and returns title, URL, and snippet fields. Search results include an untrusted-content warning because result titles and snippets can contain prompt injection.

The intended workflow is `web_search` for discovery, followed by `fetch_url` for retrieving a selected URL.

## HTTP Tool

`tools/http.py` implements `fetch_url` using only Python built-in libraries: `urllib.request`, `urllib.error`, and `urllib.parse`.

The first HTTP implementation is GET-only. It requires `http://` or `https://`, applies a timeout, caps returned text, follows default urllib redirects, reports final URL/status/content-type, and decodes using the response charset or UTF-8 fallback.

`fetch_url` does not use the auditor model; it uses direct policy permission prompts.

## Auditor Flow

Before Bash runs, `core/agent.py` calls `core/auditor.py`:

1. `classify_bash_risks(command)` adds local risk hints with regex patterns.
2. `explain_bash_command(...)` calls the configured `auditor_model`.
3. The user sees the auditor explanation, risk hints, command, cwd, timeout, and output cap.
4. The command runs only if the user approves.

The auditor prompt asks for at most 70 words. The limit is intentionally in the prompt rather than config.

## Streaming And Tool Calls

NanoHarness is stream-only. `stream_assistant_response()` consumes streamed chunks, prints content/reasoning as they arrive, and reconstructs tool calls from partial deltas.

Tool call fragments are accumulated by index until the assistant turn finishes. After that, harness executes each tool call and appends tool results to the conversation.

## Context Management And Summaries

`core/context.py` builds the model-facing context from the full saved session. Sessions still store full history, but only a bounded recent window plus optional summary is sent to the model.

Config controls `recent_messages`, `max_tool_output_chars`, `max_message_chars`, `auto_summarize`, and `summarize_after_messages`. Tool outputs and large messages are truncated for context only; the full saved session is not pruned by the context builder.

Sessions include `summary` and `summary_message_count`. `/summarize` uses `summary_model` to update the summary manually. `/summary` shows the current summary, and `/summary clear` clears both the summary text and message count. If `context.auto_summarize` is enabled, the harness summarizes after the configured message threshold and records the message count to avoid repeated summarization on every turn.

## TUI Input

`core/tui.py` uses `prompt_toolkit` for prompt editing and persistent history. History is stored in `APP_DIR/prompt_history.txt`.

This provides arrow-key navigation, editable input, and history recall without custom terminal editing code.